In [1]:
from cropspectai import config
from cropspectai.modeling.predict import predict_model
from cropspectai.data_analysis import feature_engineering
import pandas as pd


# --------------------------------------------------
# Load data
# --------------------------------------------------

#basil data
X_new = pd.read_csv(config.BASIL_EXPERIMENT_DATA_PATH / "X_new.csv", index_col = False)

#misc data
# X_new = pd.read_csv(config.PREDICTION_DATA_DIR / "misc" / "X_new.csv", index_col = False)

print("Shape of imported data: ", X_new.shape)

Shape of imported data:  (44, 15)


In [2]:
column_list = X_new.columns.to_list()
print(column_list)

['timestamp', 'specimen', 'noisy_channels', 'F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8', 'NIR']


In [3]:
AS7343_CHANNELS = list(config.AS7343_CHANNELS.keys())
print(AS7343_CHANNELS)

['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8', 'NIR']


In [4]:
non_features = column_list[:3]
print(non_features)
X_non_features = X_new[non_features].copy()
print(X_non_features.shape)

['timestamp', 'specimen', 'noisy_channels']
(44, 3)


In [5]:
X_new = X_new[AS7343_CHANNELS].copy()


In [6]:
X_new = feature_engineering.add_all_features(X_new).copy()
print(X_new.shape)

(44, 40)


In [7]:
X_new.columns

Index(['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8',
       'NIR', 'ci_red_edge_as7343', 'ci_green_as7343', 'gm_nd_as7343',
       'red_edge_ratio_as7343', 'datt2_as7343', 'datt1_as7343', 'mcari_as7343',
       'tcari_as7343', 'osavi_as7343', 'mcari_osavi_as7343',
       'tcari_osavi_as7343', 'mndvi_as7343', 'ndvi_as7343', 'gndvi_as7343',
       'sr_as7343', 'green_sr_as7343', 'evi2_as7343', 'rdvi_as7343',
       'sipi_as7343', 'psri_as7343', 'red_green_nd', 'red_edge_green_nd',
       'nir_green_nd', 'red_edge_slope_proxy', 'green_red_ratio',
       'rededge_green_ratio', 'nir_fy_ratio', 'nir_fxl_ratio'],
      dtype='str')

## import model config json file for feature list

In [8]:
import json

model_config_path = config.MODELS_DIR / "xgb_best_parameters_v_01.json"
with open(model_config_path, "r") as f:
    model_config = json.load(f)
features = model_config["features"]
print(features)

['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8', 'NIR', 'mndvi_as7343', 'sipi_as7343', 'ci_red_edge_as7343', 'psri_as7343', 'red_green_nd', 'datt1_as7343', 'mcari_osavi_as7343']


### remove columns not in feature list

In [9]:
X_new = X_new[features].copy()
X_new.columns

Index(['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8',
       'NIR', 'mndvi_as7343', 'sipi_as7343', 'ci_red_edge_as7343',
       'psri_as7343', 'red_green_nd', 'datt1_as7343', 'mcari_osavi_as7343'],
      dtype='str')

In [15]:
X_new.tail(20)

,F1,F2,FZ,F3,F4,FY,F5,FXL,F6,F7,F8,NIR,mndvi_as7343,sipi_as7343,ci_red_edge_as7343,psri_as7343,red_green_nd,datt1_as7343,mcari_osavi_as7343
24,0.0,0.013208,0.007480,0.007576,0.072262,0.083818,0.121622,0.064282,0.026451,0.072422,0.333333,0.125000,0.288159,2.235155,-0.625000,0.000480,-0.253551,-3.962361,5.895426
25,0.0,0.015094,0.007874,0.007576,0.072262,0.083818,0.121622,0.064282,0.026451,0.072422,0.333333,0.125000,0.289409,2.227662,-0.625000,0.000480,-0.253551,-3.962361,5.895426
26,0.0,0.015094,0.007874,0.007576,0.072262,0.083818,0.120946,0.064282,0.026451,0.072422,0.333333,0.125000,0.289409,2.227662,-0.625000,0.000480,-0.250941,-3.962361,5.891779
27,0.0,0.013208,0.007874,0.007576,0.072262,0.083818,0.121622,0.064282,0.026451,0.072422,0.333333,0.125000,0.289409,2.227662,-0.625000,0.000480,-0.253551,-3.962361,5.895426
28,0.0,0.015094,0.007874,0.007576,0.072262,0.083818,0.121622,0.064282,0.026451,0.072422,0.333333,0.125000,0.289409,2.227662,-0.625000,0.000480,-0.253551,-3.962361,5.895426
29,0.0,0.000000,0.000000,0.000000,0.025014,0.029935,0.047297,0.021427,0.004959,0.037659,0.205128,0.041667,0.050526,10.395958,-0.796873,0.061644,-0.113447,-40.783683,38.105595
30,0.0,0.000000,0.000000,0.000000,0.025014,0.029080,0.047297,0.021427,0.004959,0.034762,0.179487,0.041667,0.090345,6.034323,-0.767855,0.054310,-0.152756,-19.959450,18.027867
31,0.0,0.000000,0.000000,0.000000,0.025014,0.029080,0.047297,0.021427,0.004959,0.034762,0.179487,0.041667,0.090345,6.034323,-0.767855,0.054310,-0.152756,-19.959450,18.027867
32,0.0,0.000000,0.000000,0.000000,0.024736,0.029080,0.047297,0.020933,0.004959,0.033024,0.179487,0.041667,0.115717,4.820896,-0.767855,0.046176,-0.177699,-15.945852,15.270349
33,0.0,0.000000,0.000000,0.000000,0.025014,0.029080,0.047297,0.020274,0.004959,0.031866,0.179487,0.041667,0.133287,4.251301,-0.767855,0.038175,-0.194927,-14.061830,14.020614


## load best xgb model

In [11]:
model_path = config.MODELS_DIR / "xgb_v_01.joblib"

In [12]:
predictions = predict_model(X_new, model_path)
print(predictions)

[19.325014 22.56456  25.53629  23.261965 24.660383 22.603994 21.81295
 21.640808 49.590717 37.080364 23.773594 24.58659  23.75407  23.797398
 18.981905 18.053276 22.039793 22.039793 17.899874 17.700214 20.414234
 17.783699 17.654444 18.9075   18.450342 18.55969  18.55969  18.449728
 18.55969  22.356148 22.397186 22.397186 22.432526 22.397186 18.158945
 17.979061 18.322674 18.322674 18.061716 20.457312 20.699482 20.053308
 19.853043 19.721188]


In [13]:
df_predict = X_non_features[non_features].copy()
df_predict["Cab (µg/cm²)"] = predictions

In [14]:
df_predict

,timestamp,specimen,noisy_channels,Cab (µg/cm²)
0,2026-10-10T10:57:57,sun_water,F3 (129.1%),19.325014
1,2026-10-10T10:58:19,sun_water,FZ (14.3%),22.564560
2,2026-10-10T10:58:52,sun_water,F3 (9.6%); NIR (28.4%),25.536289
3,2026-10-10T11:00:51,sun_no_water,FZ (6.5%); F8 (5.4%),23.261965
4,2026-10-10T11:01:20,sun_no_water,NaN,24.660383
5,2026-10-10T11:02:06,sun_no_water,F2 (86.1%),22.603994
6,2026-10-10T11:04:17,dark_water,NaN,21.812950
7,2026-10-10T11:04:32,dark_water,NaN,21.640808
8,2026-10-10T11:04:55,dark_water,F2 (69.0%); F3 (10.9%); F7 (7.0%),49.590717
9,2026-10-10T11:05:35,dark_no_water,NaN,37.080364
